# Project 7: richer since 2010, in which currency?

**The question.** Has the average Surinamese person become richer since 2010, measured in US dollars and in Surinamese dollars (SRD)?

**The data.** Two World Bank tables for Suriname, both in current prices:

- GDP per person in current US dollars (indicator NY.GDP.PCAP.CD), as the World Bank's CSV download lays it out: one row, with a column for every year from 1960 to 2025.
- GDP per person in current Surinamese dollars (indicator NY.GDP.PCAP.CN), from the World Bank's API: one row for every year from 2010 to 2025.

The World Bank last updated both on 2026-10-08. The course copies were downloaded on 2026-11-06.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker counts the rows of every table before and after each join, and the AI checker tests every join the AI writes against those counts.

**What you hand in on Friday.** This notebook with every CHECK passing, two charts, a note of about 150 words written for a family, and your AI log.

## Load the data

This cell is given. It reads both tables from the course folder, or straight from the World Bank if the files are not there.

In [ ]:
import io, json, urllib.request, zipfile
import pandas as pd

USD_URL = "https://api.worldbank.org/v2/country/SUR/indicator/NY.GDP.PCAP.CD?downloadformat=csv"
SRD_URL = ("https://api.worldbank.org/v2/country/SUR/indicator/NY.GDP.PCAP.CN"
           "?format=json&date=2010:2025&per_page=100")

# GDP per person in US dollars: the World Bank's CSV download, one column per year
try:
    usd_wide = pd.read_csv("../data/week07-gdp-per-person-usd.csv")
except FileNotFoundError:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(USD_URL).read()))
    name = [n for n in z.namelist() if n.startswith("API_")][0]
    usd_wide = pd.read_csv(z.open(name), skiprows=4).dropna(axis=1, how="all")

# GDP per person in Surinamese dollars: the World Bank's API, one row per year
try:
    srd = pd.read_csv("../data/week07-gdp-per-person-srd.csv")
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(SRD_URL))[1]
    srd = pd.DataFrame(
        [{"year": int(r["date"]), "gdp_srd": r["value"]} for r in raw if r["value"] is not None]
    ).sort_values("year").reset_index(drop=True)

print("usd_wide:", usd_wide.shape, "  srd:", srd.shape)

## TODO 1: describe both tables

Write code that prints, for each table, the number of rows and columns, the first and last year, the number of missing values and the type of the year. In `usd_wide` the years are column names rather than values in a column, so look at `usd_wide.columns`.

In [ ]:
# TODO 1

## TODO 2: from wide to long

Make a table called `usd` with two columns, `year` and `gdp_usd`, and one row for each year column of `usd_wide`. The method `melt` does this.

Print the type of `year` in `usd` and in `srd`. Write both types in your AI log, because you will need them in TODO 3.

In [ ]:
# TODO 2
usd = None

In [ ]:
# CHECK 2
assert usd is not None, "usd is still empty"
assert {"year", "gdp_usd"} <= set(usd.columns), "usd needs the columns year and gdp_usd"
year_cols = [c for c in usd_wide.columns if str(c).isdigit()]
assert len(usd) == len(year_cols), "usd needs one row for each year column of usd_wide"
assert usd["year"].astype(int).is_unique, "a year appears more than once"
print("CHECK 2 passed:", len(usd), "rows; year is", usd["year"].dtype, "in usd and", srd["year"].dtype, "in srd")

## TODO 3: join the two tables (the AI check)

Make a table called `both` with the columns `year`, `gdp_usd` and `gdp_srd`, one row for each year from 2010 to 2025, and `year` stored as a whole number.

First write in your AI log how many rows `usd` and `srd` have, and how many rows `both` should have. Then give your AI assistant this request: *"I have two pandas tables. usd has the columns year and gdp_usd, one row per year from 1960 to 2025. srd has the columns year and gdp_srd, one row per year from 2010 to 2025. Write code that joins them into a table called both, with one row per year from 2010 to 2025."*

Paste its code below, run it and count the rows of `both`. If pandas refuses the join, show the AI the error message and test its next answer in the same way. Record every attempt in the AI log: the rows you expected, the rows you got, and any year that was lost or repeated.

In [ ]:
# TODO 3
both = None

In [ ]:
# CHECK 3
assert both is not None, "both is still empty"
assert pd.api.types.is_integer_dtype(both["year"]), "year in both must be a whole number, not text"
assert both["year"].is_unique, "a year appears more than once"
assert sorted(both["year"]) == sorted(srd["year"].astype(int)), "both must have exactly the years of the SRD table"
assert both[["gdp_usd", "gdp_srd"]].notna().all().all(), "some values are missing"
u = usd_wide.iloc[0]
s = srd.assign(year=srd["year"].astype(int)).set_index("year")["gdp_srd"]
assert all(abs(r.gdp_usd - float(u[str(r.year)])) < 1e-6 and abs(r.gdp_srd - s[r.year]) < 1e-6
           for r in both.itertuples()), "a value sits next to the wrong year"
print("CHECK 3 passed:", len(both), "rows, from", both["year"].min(), "to", both["year"].max())

## TODO 4: how many times larger?

Store in `times_usd` how many times larger GDP per person in US dollars was in 2025 than in 2010. Store the same for Surinamese dollars in `times_srd`.

In [ ]:
# TODO 4
times_usd = None
times_srd = None

In [ ]:
# CHECK 4
b = both.set_index("year")
assert times_usd is not None and times_srd is not None, "times_usd or times_srd is still empty"
assert abs(times_usd - b.loc[2025, "gdp_usd"] / b.loc[2010, "gdp_usd"]) < 0.01
assert abs(times_srd - b.loc[2025, "gdp_srd"] / b.loc[2010, "gdp_srd"]) < 0.01
print("CHECK 4 passed:", round(times_usd, 2), "in US$ and", round(times_srd, 2), "in SRD")

## TODO 5: the exchange rate inside the figures

Both tables measure the same thing in two currencies. Dividing the SRD figure by the US-dollar figure therefore gives the number of SRD the World Bank counted for one US dollar in that year. Add this to `both` as a column called `srd_per_usd`. Store in `times_rate` how many times larger it was in 2025 than in 2010.

Then print `times_usd * times_rate` next to `times_srd`. Your note must explain in one sentence why the two agree.

In [ ]:
# TODO 5
times_rate = None

In [ ]:
# CHECK 5
assert "srd_per_usd" in both.columns, "both has no column srd_per_usd"
b = both.set_index("year")
assert (abs(b["srd_per_usd"] - b["gdp_srd"] / b["gdp_usd"]) < 1e-9).all()
assert times_rate is not None, "times_rate is still empty"
assert abs(times_rate - b.loc[2025, "srd_per_usd"] / b.loc[2010, "srd_per_usd"]) < 0.01
print("CHECK 5 passed:", round(times_rate, 2))

## TODO 6: the years that went down

Store in `usd_down_years` a list of the years in which GDP per person in US dollars was lower than the year before. Store the same for Surinamese dollars in `srd_down_years`. A list may be empty.

In [ ]:
# TODO 6
usd_down_years = None
srd_down_years = None

In [ ]:
# CHECK 6
assert usd_down_years is not None and srd_down_years is not None, "a list is still empty"
b = both.set_index("year").sort_index()
exp_usd = [int(y) for y in b.index[1:] if b.loc[y, "gdp_usd"] < b.loc[y - 1, "gdp_usd"]]
exp_srd = [int(y) for y in b.index[1:] if b.loc[y, "gdp_srd"] < b.loc[y - 1, "gdp_srd"]]
assert sorted(int(y) for y in usd_down_years) == exp_usd, "usd_down_years is not right"
assert sorted(int(y) for y in srd_down_years) == exp_srd, "srd_down_years is not right"
print("CHECK 6 passed:", exp_usd, "in US$ and", exp_srd, "in SRD")

## TODO 7: two charts side by side

Draw GDP per person by year as two line charts side by side, one in US dollars and one in Surinamese dollars. Give each chart a title, label both axes, start both value axes at zero and name the World Bank as the source.

In [ ]:
# TODO 7

## TODO 8: the note for a family

In a text cell below, write about 150 words for a family that asks whether the average person in Suriname is richer than in 2010. Answer in both currencies with the numbers you computed, explain why the two answers differ, name the source, and say one thing the data cannot tell them.